# **Evaluation**

## **GridSearchCV was applied twice:**
### **Stage 1: coarse search over a wide parameter range using 5-fold CV**
### **Stage 2: fine search over a narrower range centered on the best results from Stage 1.**

In [1]:
!pip install xgboost scikit-learn

import csv
import json
import math
import numpy as np
import xgboost as xgb

from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# directory path
FOLDER_PATH = "/content/drive/MyDrive/Colab_Notebooks/DSCI553/Project"
if not FOLDER_PATH.endswith("/"):
    FOLDER_PATH += "/"

TRAIN_FILE = FOLDER_PATH + "data/yelp_train.csv"
USER_FILE = FOLDER_PATH + "data/user.json"
BUSINESS_FILE = FOLDER_PATH + "data/business.json"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
def count_list_field(s):
    # "a,b,c" => 3; None / "None" / "" => 0
    if not s or s == "None":
        return 0
    return len(str(s).split(","))

def get_log1p(x):
    try:
        x = float(x)
    except Exception:
        return 0.0
    if x < 0:
        return 0.0
    return math.log1p(x)

def get_user_features_obj(user):
    try:
        elite = user.get("elite", "None")
        elite_len = count_list_field(elite)
        friends = user.get("friends", "None")
        friends_cnt = count_list_field(friends)
        return {"u_avg_stars": float(user.get("average_stars", float("nan"))),
                "u_review_cnt": int(user.get("review_count", 0)),
                "u_fans": int(user.get("fans", 0)),
                "u_useful": int(user.get("useful", 0)),
                "u_cool": int(user.get("cool", 0)),
                "u_funny": int(user.get("funny", 0)),
                "u_friends_cnt": friends_cnt,
                "u_elite_years": elite_len,
                "u_cmp_hot": int(user.get("compliment_hot", 0)),
                "u_cmp_cool": int(user.get("compliment_cool", 0)),
                "u_cmp_funny": int(user.get("compliment_funny", 0)),
                "u_cmp_photos": int(user.get("compliment_photos", 0))}
    except Exception:
        return {}

def get_bool(attribute_dict, key):
    # convert True/False/None to 1/0
    if not isinstance(attribute_dict, dict):
        return 0
    v = attribute_dict.get(key)
    if v is None:
        return 0
    s = str(v).strip().lower()
    return 1 if s in ["true", "1", "yes"] else 0

def get_price(attribute_dict):
    # 1~4, missing value = 2
    if not isinstance(attribute_dict, dict):
        return 2
    v = attribute_dict.get("RestaurantsPriceRange2")
    try:
        return int(v)
    except Exception:
        return 2

def get_business_features_obj(business):
    try:
        attrs = business.get("attributes", {}) or {}
        categories = business.get("categories", "") or ""
        is_restaurant = 1 if "Restaurant" in categories else 0
        return {"b_avg_stars": float(business.get("stars", float('nan'))),
                "b_review_cnt": int(business.get("review_count", 0)),
                "b_latitude": float(business.get("latitude", 0.0)),
                "b_longitude": float(business.get("longitude", 0.0)),
                "b_price": float(get_price(attrs)),
                "b_credit_card": float(get_bool(attrs, "BusinessAcceptsCreditCards")),
                "b_reservation": float(get_bool(attrs, "RestaurantsReservations")),
                "b_table_service": float(get_bool(attrs, "RestaurantsTableService")),
                "b_wheelchair": float(get_bool(attrs, "WheelchairAccessible")),
                "b_is_restaurant": float(is_restaurant)}
    except Exception:
        return {}

# competition.py => get_features() without Spark / broadcast
def build_feature_vector(user_id, business_id, user_feature_dict, business_feature_dict):
    uf = user_feature_dict.get(user_id, {})
    bf = business_feature_dict.get(business_id, {})

    features = [
        # business side
        float(bf.get("b_avg_stars", 0.0)),
        get_log1p(bf.get("b_review_cnt", 0)),
        float(bf.get("b_latitude", 0.0)),
        float(bf.get("b_longitude", 0.0)),
        float(bf.get("b_price", 0.0)),
        float(bf.get("b_credit_card", 0.0)),
        float(bf.get("b_reservation", 0.0)),
        float(bf.get("b_table_service", 0.0)),
        float(bf.get("b_wheelchair", 0.0)),
        float(bf.get("b_is_restaurant", 0.0)),
        # user side
        float(uf.get("u_avg_stars", 0.0)),
        get_log1p(uf.get("u_review_cnt", 0)),
        get_log1p(uf.get("u_fans", 0)),
        get_log1p(uf.get("u_useful", 0)),
        get_log1p(uf.get("u_cool", 0)),
        get_log1p(uf.get("u_funny", 0)),
        get_log1p(uf.get("u_friends_cnt", 0)),
        get_log1p(uf.get("u_elite_years", 0)),
        get_log1p(uf.get("u_cmp_hot", 0)),
        get_log1p(uf.get("u_cmp_cool", 0)),
        get_log1p(uf.get("u_cmp_funny", 0)),
        get_log1p(uf.get("u_cmp_photos", 0)),
    ]
    features = [0.0 if (not math.isfinite(f)) else f for f in features]
    return features

In [4]:
#  2-stage GridSearch

def main_two_stage():
    # load user.json
    print("Loading user.json ...")
    user_feature_dict = {}
    with open(USER_FILE, "r", encoding = "utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            u = json.loads(line)
            user_feature_dict[u["user_id"]] = get_user_features_obj(u)
    # load business.json
    print("Loading business.json ...")
    business_feature_dict = {}
    with open(BUSINESS_FILE, "r", encoding = "utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            b = json.loads(line)
            business_feature_dict[b["business_id"]] = get_business_features_obj(b)
    # load yelp_train.csv
    print("Loading yelp_train.csv and building feature matrix ...")
    X = []
    y = []
    with open(TRAIN_FILE, "r", encoding = "utf-8") as f:
        reader = csv.reader(f)
        header = next(reader)    # user_id, business_id, stars
        for row in reader:
            if len(row) < 3:
                continue
            u, b, rating = row[0], row[1], float(row[2])
            feat = build_feature_vector(u, b, user_feature_dict, business_feature_dict)
            X.append(feat)
            y.append(rating)

    X = np.array(X, dtype = float)
    y = np.array(y, dtype = float)
    print("Total training samples:", X.shape[0])

    # subsample for GridSearch
    max_samples = 80000
    if X.shape[0] > max_samples:
        idx = np.random.choice(X.shape[0], size = max_samples, replace = False)
        X = X[idx]
        y = y[idx]
        print("Subsampled to:", X.shape[0])

    # split train & valid
    X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size = 0.2, random_state = 42)

    # Stage 1: coarse search (5-fold CV)

    print("\nStage 1: Coarse GridSearch (5-fold)")

    param_grid_stage1 = {"learning_rate": [0.03, 0.05, 0.07],
                         "max_depth": [4, 5, 6],
                         "n_estimators": [600, 800, 1000],
                         "subsample": [0.7, 0.8],
                         "colsample_bytree": [0.6, 0.7, 0.8],
                         "min_child_weight": [2, 4],
                         "reg_alpha": [0.0, 0.05, 0.1],
                         "reg_lambda": [1.0]}

    xgb_base = xgb.XGBRegressor(objective = "reg:squarederror", nthread = 4, tree_method = "hist", random_state = 42)

    grid1 = GridSearchCV(estimator = xgb_base, param_grid = param_grid_stage1, scoring = "neg_mean_squared_error", cv = 5, verbose = 2, n_jobs = -1)

    grid1.fit(X_train, y_train)

    best_params_1 = grid1.best_params_
    best_rmse_1 = math.sqrt(-grid1.best_score_)

    print("\n[Stage 1] Best params:")
    print(best_params_1)
    print("[Stage 1] Best CV RMSE:", best_rmse_1)

    # Stage 2: fine search (5-fold CV)

    print("\nStage 2: Fine GridSearch (5-fold)")

    center_lr = best_params_1.get("learning_rate", 0.05)
    center_md = best_params_1.get("max_depth", 5)
    center_ne = best_params_1.get("n_estimators", 1000)
    center_mcw = best_params_1.get("min_child_weight", 4)
    center_ra = best_params_1.get("reg_alpha", 0.05)

    def around(val, choices):
        # select one or two values from choices that are closest to val, used to narrow the range
        s = sorted(choices)
        if val in s:
            return [val]
        # otherwise, pick the two closest values
        s2 = sorted(s, key = lambda x: abs(x - val))
        return sorted(list(set(s2[:2])))

    lr_candidates = around(center_lr, [0.03, 0.04, 0.05, 0.06])
    md_candidates = around(center_md, [4, 5, 6])
    ne_candidates = around(center_ne, [800, 900, 1000, 1100])
    mcw_candidates = around(center_mcw, [3, 4, 5])
    ra_candidates = around(center_ra, [0.0, 0.05, 0.1])

    print("Stage 2 search space:")
    print("  learning_rate:", lr_candidates)
    print("  max_depth:", md_candidates)
    print("  n_estimators:", ne_candidates)
    print("  min_child_weight:", mcw_candidates)
    print("  reg_alpha:", ra_candidates)

    param_grid_stage2 = {"learning_rate": lr_candidates,
                         "max_depth": md_candidates,
                         "n_estimators": ne_candidates,
                         "subsample": [0.8],    # fixed 0.8
                         "colsample_bytree": [0.7],    # fixed 0.7
                         "min_child_weight": mcw_candidates,
                         "reg_alpha": ra_candidates,
                         "reg_lambda": [1.0]}    # fixed 1.0

    grid2 = GridSearchCV(estimator = xgb_base, param_grid = param_grid_stage2, scoring = "neg_mean_squared_error", cv = 5, verbose = 2, n_jobs = -1)

    grid2.fit(X_train, y_train)

    best_params_2 = grid2.best_params_
    best_rmse_2 = math.sqrt(-grid2.best_score_)

    print("\n[Stage 2] Best params:")
    print(best_params_2)
    print("[Stage 2] Best CV RMSE:", best_rmse_2)

    # sanity check on holdout
    best_model = grid2.best_estimator_
    y_pred_valid = best_model.predict(X_valid)
    valid_rmse = math.sqrt(mean_squared_error(y_valid, y_pred_valid))
    print("[Stage 2] Holdout validation RMSE:", valid_rmse)

# run
main_two_stage()

Loading user.json ...
Loading business.json ...
Loading yelp_train.csv and building feature matrix ...
Total training samples: 455854
Subsampled to: 80000

Stage 1: Coarse GridSearch (5-fold)
Fitting 5 folds for each of 972 candidates, totalling 4860 fits

[Stage 1] Best params:
{'colsample_bytree': 0.6, 'learning_rate': 0.03, 'max_depth': 4, 'min_child_weight': 2, 'n_estimators': 600, 'reg_alpha': 0.1, 'reg_lambda': 1.0, 'subsample': 0.8}
[Stage 1] Best CV RMSE: 0.9848831076054259

Stage 2: Fine GridSearch (5-fold)
Stage 2 search space:
  learning_rate: [0.03]
  max_depth: [4]
  n_estimators: [800, 900]
  min_child_weight: [3, 4]
  reg_alpha: [0.1]
Fitting 5 folds for each of 4 candidates, totalling 20 fits

[Stage 2] Best params:
{'colsample_bytree': 0.7, 'learning_rate': 0.03, 'max_depth': 4, 'min_child_weight': 3, 'n_estimators': 800, 'reg_alpha': 0.1, 'reg_lambda': 1.0, 'subsample': 0.8}
[Stage 2] Best CV RMSE: 0.9856377944897016
[Stage 2] Holdout validation RMSE: 0.97530043198623

#### For the model-based part, I followed a two-stage hyperparameter tuning strategy.
#### In Stage 1, I performed a coarse GridSearchCV over a wide parameter range using 5-fold cross-validation on a randomly subsampled subset of the training data (around 80,000 samples) due to computational constraints on Colab.
#### In Stage 2, I refined the search in a smaller neighborhood centered around the best configuration from Stage 1 (including learning_rate, max_depth, n_estimators, min_child_weight, and reg_alpha) again with 5-fold CV.
#### Several configurations achieved very similar cross-validation RMSE. The final hyperparameters used in the hybrid system were selected based on the performance of the full CF+XGBoost pipeline on the validation set, not only on the standalone XGBoost CV score.

# **Runtime & Error Distribution**

In [5]:
from google.colab import drive

drive.mount('/content/drive')

folder_path = "/content/drive/MyDrive/Colab_Notebooks/DSCI553/Project"
competition_path = folder_path + "/competition.py"
val_file = "yelp_val.csv"
pred_file = "val_prediction.csv"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import time

start_time = time.time()

!python "{competition_path}" "{folder_path}" "{val_file}" "{pred_file}"

end_time = time.time()
print("Execution Time (seconds):", end_time - start_time)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/18 22:52:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [22:53:42] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  bst.update(dtrain, iteration=i, fobj=obj)
Execution Time (seconds): 99.53652501106262


In [7]:
import csv
import math

folder_path = "/content/drive/MyDrive/Colab_Notebooks/DSCI553/Project"
val_file = folder_path + "/data/yelp_val.csv"
pred_file = folder_path + "/output/val_prediction.csv"

# ground truth
true_dict = {}
with open(val_file, "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    header = next(reader)    # user_id, business_id, stars
    for row in reader:
        if len(row) < 3:
            continue
        u, b, r = row[0], row[1], float(row[2])
        true_dict[(u, b)] = r

# prediction
pred_dict = {}
with open(pred_file, "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    header = next(reader)    # user_id, business_id, prediction
    for row in reader:
        if len(row) < 3:
            continue
        u, b, p = row[0], row[1], float(row[2])
        pred_dict[(u, b)] = p

# calculate error bins + RMSE
N0 = N1 = N2 = N3 = N4 = 0
sq_err_sum = 0.0
n = 0

for key, true_rating in true_dict.items():
    if key not in pred_dict:
        continue
    pred_rating = pred_dict[key]
    err = abs(pred_rating - true_rating)
    sq_err_sum += err * err
    n += 1

    if err < 1:
        N0 += 1
    elif err < 2:
        N1 += 1
    elif err < 3:
        N2 += 1
    elif err < 4:
        N3 += 1
    else:
        N4 += 1

rmse = math.sqrt(sq_err_sum / n)
print("Error Distribution on validation (yelp_val.csv):")
print(">=0 and <1:", N0)
print(">=1 and <2:", N1)
print(">=2 and <3:", N2)
print(">=3 and <4:", N3)
print(">=4       :", N4)
print("\nRMSE:", rmse)

Error Distribution on validation (yelp_val.csv):
>=0 and <1: 102076
>=1 and <2: 33047
>=2 and <3: 6155
>=3 and <4: 766
>=4       : 0

RMSE: 0.9781750475325418
